In [2]:
import pandas as pd
import pickle
import numpy as np
import json

# Charger les références du train
train_medians = pd.read_csv('data/clean/train_medians.csv', index_col=0).squeeze()
with open('data/clean/cols_to_drop.json', 'r') as f:
    cols_to_drop = json.load(f)

# Charger le modèle et les colonnes du train
with open('final_model.pkl', 'rb') as f:
    final_model = pickle.load(f)

X_train = pd.read_csv('data/clean/X_train.csv')
train_columns = X_train.columns.tolist()

# Charger le test Kaggle
test = pd.read_csv('./data/application_test.csv')
test_ids = test['SK_ID_CURR']

# Appliquer les mêmes transformations
test = test.drop(columns=[col for col in cols_to_drop if col in test.columns])
test['DAYS_EMPLOYED'] = test['DAYS_EMPLOYED'].replace(365243, np.nan)
test = test.drop(columns=['WEEKDAY_APPR_PROCESS_START', 'SK_ID_CURR'], errors='ignore')

# Imputer avec les médianes du TRAIN
num_cols_test = test.select_dtypes(include=['float64', 'int64']).columns
for col in num_cols_test:
    if test[col].isnull().sum() > 0:
        test[col] = test[col].fillna(train_medians.get(col, test[col].median()))

# Catégorielles
cat_cols_test = test.select_dtypes(include=['object']).columns
for col in cat_cols_test:
    test[col] = test[col].fillna('Unknown')

# One-hot encoding
test = pd.get_dummies(test, drop_first=True)

# Aligner les colonnes avec le train
test = test.reindex(columns=train_columns, fill_value=0)

print(test.shape)

(48744, 178)


In [3]:
# Prédire
y_pred_test = final_model.predict_proba(test)[:, 1]

# Générer le fichier de submission
submission = pd.DataFrame({
    'SK_ID_CURR': test_ids,
    'TARGET': y_pred_test
})

submission.to_csv('data/clean/submission.csv', index=False)
print(submission.head())
print(f"Shape : {submission.shape}")

   SK_ID_CURR    TARGET
0      100001  0.022813
1      100005  0.068933
2      100013  0.026577
3      100028  0.082294
4      100038  0.099493
Shape : (48744, 2)
